# 05b — WBIC Summary (Trial-Normalized)

Loads per-session WBIC CSVs produced by `04_wbic_fitting.ipynb` and
computes a trial-count-normalized summary.

**Pipeline**
1. Load `results/WBIC/WBIC_{model}_{stage}.csv` (one row per session)
2. Compute `wbic_per_trial = WBIC / n_trials` for each session
3. Per subject: mean ± SD of `wbic_per_trial` (and each parameter)
4. Save to `results/WBIC/WBIC_norm_subj_{model}_{stage}.csv`

**Why normalize by n_trials?**
WBIC scales linearly with session length.  Dividing by `n_trials` makes
the metric comparable across sessions of different lengths and allows
meaningful averaging across subjects that may differ in trial count.

The per-session CSVs (with raw WBIC, n_trials, and all parameter
posterior means) are also kept for downstream simulation use.

In [ ]:
# ── Cell 1: Configuration ─────────────────────────────────────────────────

MODELS_TO_SUMMARIZE = ['mf_p', 'rac_p', 'hyb_p', 'ls', 'ls_asym_p', 'hyb_inf']
# MODELS_TO_SUMMARIZE = ['mf_p']
STAGES = ['4.2', '4.3', '4.4', '4.5', '4.6', '4.7', '4.8']

# Set True to recompute even if the summary CSV already exists
FORCE_RECOMPUTE = False

In [ ]:
# ── Cell 2: Imports & path setup ──────────────────────────────────────────

import sys, os

_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    _organized = os.path.dirname(_here)
else:
    _organized = os.path.join(_here, 'organized')
    if not os.path.isdir(_organized):
        _organized = _here
sys.path.insert(0, _organized)
sys.path.insert(0, os.path.join(_organized, 'src'))

import numpy as np
import pandas as pd

from config import (
    SUBJECTS, WBIC_RESULTS_DIR, FIGURES_DIR,
    wbic_result_csv, wbic_norm_subj_csv
)
from src.wbic import WBIC_MODELS

print('Organized root:', _organized)
print('WBIC results dir:', WBIC_RESULTS_DIR)

In [ ]:
# ── Cell 3: Load per-session CSVs, add WBIC/n_trials column ───────────────
#
# Reads every available (model × stage) CSV and stacks them into df_all.
# Adds the column wbic_per_trial = WBIC / n_trials.

dfs = []
for model_key in MODELS_TO_SUMMARIZE:
    for stage in STAGES:
        csv_path = wbic_result_csv(model_key, stage)
        if not os.path.exists(csv_path):
            continue
        tmp = pd.read_csv(csv_path)
        dfs.append(tmp)

if not dfs:
    raise FileNotFoundError(
        'No per-session WBIC CSVs found. '
        'Run 04_wbic_fitting.ipynb first.')

df_all = pd.concat(dfs, ignore_index=True)

# pandas reads stage as float (e.g., 4.2); force string for comparisons below
df_all['stage'] = df_all['stage'].astype(str)

# Normalize WBIC by trial count
df_all['wbic_per_trial'] = df_all['WBIC'] / df_all['n_trials']

print(f'Loaded {len(df_all)} session rows across '
      f'{df_all["model"].nunique()} model(s) and '
      f'{df_all["stage"].nunique()} stage(s).')
print()
print('Session counts by model × stage:')
print(df_all.groupby(['model', 'stage']).size().unstack(fill_value=0).to_string())

In [ ]:
# ── Cell 4: Session-level summary table ───────────────────────────────────
#
# Display WBIC per trial stats (mean ± SD) across sessions per model×stage,
# as a quick sanity check before saving subject-level CSVs.

summary = (
    df_all
    .groupby(['model', 'stage'])['wbic_per_trial']
    .agg(n_sessions='count', mean='mean', std='std')
    .reset_index()
)
summary['mean±std'] = (
    summary['mean'].round(4).astype(str) + ' ± ' +
    summary['std'].round(4).astype(str)
)
print('WBIC per trial (all sessions pooled):')
print(summary[['model', 'stage', 'n_sessions', 'mean±std']].to_string(index=False))

In [ ]:
# ── Cell 5: Per-subject mean ± SD, save summary CSVs ─────────────────────
#
# For each (model, stage):
#   1. Group sessions by subject
#   2. Compute mean and SD of wbic_per_trial and each parameter
#   3. Save to WBIC_norm_subj_{model}_{stage}.csv
#
# Output columns:
#   subject, n_sessions, n_trials_total,
#   wbic_per_trial_mean, wbic_per_trial_std,
#   WBIC_mean, WBIC_std,
#   {param}_mean, {param}_std  (one pair per parameter)

for model_key in MODELS_TO_SUMMARIZE:
    param_names = WBIC_MODELS[model_key].param_names
    value_cols = ['wbic_per_trial', 'WBIC'] + list(param_names)

    for stage in STAGES:
        out_path = wbic_norm_subj_csv(model_key, stage)

        if not FORCE_RECOMPUTE and os.path.exists(out_path):
            print(f'[{model_key} | stage {stage}] Already exists — skipping.')
            continue

        mask = (df_all['model'] == model_key) & (df_all['stage'] == stage)
        df = df_all[mask]
        if df.empty:
            continue

        # Aggregate: mean and SD per subject
        agg = (
            df.groupby('subject')[value_cols]
            .agg(['mean', 'std'])
            .reset_index()
        )
        # Flatten multi-level column names
        agg.columns = ['subject'] + [
            f'{col}_{stat}'
            for col in value_cols
            for stat in ('mean', 'std')
        ]

        # Add session and trial counts per subject
        counts = df.groupby('subject').agg(
            n_sessions=('WBIC', 'count'),
            n_trials_total=('n_trials', 'sum')
        ).reset_index()
        agg = counts.merge(agg, on='subject')

        agg.to_csv(out_path, index=False)
        print(f'[{model_key} | stage {stage}] Saved: {out_path}')

print('\nDone.')

In [ ]:
# ── Cell 6: Session-level parameter timeline ──────────────────────────────
#
# For a quick visual inspection: show per-session parameter values over
# time (session_idx) for each subject, for the first model in the list.
# Useful for spotting parameter drift or session-to-session variability
# before running per-session simulations.

import matplotlib.pyplot as plt

INSPECT_MODEL = MODELS_TO_SUMMARIZE[0]  # change as needed
INSPECT_STAGE = STAGES[-2]              # e.g. stage 4.7

mask = (df_all['model'] == INSPECT_MODEL) & (df_all['stage'] == INSPECT_STAGE)
df_inspect = df_all[mask].copy()

if df_inspect.empty:
    print(f'No data for model={INSPECT_MODEL}, stage={INSPECT_STAGE}.')
else:
    param_names = list(WBIC_MODELS[INSPECT_MODEL].param_names)
    plot_cols = ['wbic_per_trial'] + param_names
    n_cols_plot = len(plot_cols)

    fig, axes = plt.subplots(
        n_cols_plot, 1,
        figsize=(10, 2.5 * n_cols_plot),
        sharex=False
    )
    if n_cols_plot == 1:
        axes = [axes]

    for ax, col in zip(axes, plot_cols):
        for subj in SUBJECTS:
            subj_data = df_inspect[df_inspect['subject'] == subj].sort_values('date')
            if subj_data.empty:
                continue
            ax.plot(range(len(subj_data)), subj_data[col].values,
                    marker='o', markersize=4, label=subj, linewidth=1.2)
        ax.set_ylabel(col, fontsize=9)
        ax.spines[['top', 'right']].set_visible(False)

    axes[-1].set_xlabel('Session index (within stage)', fontsize=9)
    axes[0].legend(fontsize=8, ncol=3, frameon=False)
    axes[0].set_title(
        f'{INSPECT_MODEL} | stage {INSPECT_STAGE} — per-session parameters',
        fontsize=11
    )
    fig.tight_layout()
    plt.show()

In [ ]:
# ── Cell 7: Progress table ─────────────────────────────────────────────────
#
# Shows which WBIC_norm_subj CSVs exist.

print(f"{'Model':<12}", end='')
for stage in STAGES:
    print(f"{stage:>6}", end='')
print()
print('-' * (12 + 6 * len(STAGES)))

for model_key in MODELS_TO_SUMMARIZE:
    print(f"{model_key:<12}", end='')
    for stage in STAGES:
        path = wbic_norm_subj_csv(model_key, stage)
        symbol = ' OK ' if os.path.exists(path) else ' -- '
        print(f"{symbol:>6}", end='')
    print()

In [ ]:
# ── Cell 8: Average WBIC per stage (all models) ───────────────────────────
#
# Loads every available per-session WBIC CSV, computes mean wbic_per_trial
# per (model × stage), and plots one line per model.

import glob
import matplotlib.pyplot as plt

# Load all per-session CSVs (exclude subj/norm summaries)
csv_files = sorted(glob.glob(os.path.join(WBIC_RESULTS_DIR, 'WBIC_*.csv')))
csv_files = [f for f in csv_files
             if not os.path.basename(f).startswith('WBIC_subj_')
             and not os.path.basename(f).startswith('WBIC_norm_')]

dfs = []
for f in csv_files:
    try:
        tmp = pd.read_csv(f)
        dfs.append(tmp)
    except Exception as e:
        print(f'  Warning: could not read {f}: {e}')

df = pd.concat(dfs, ignore_index=True)
df['wbic_per_trial'] = df['WBIC'] / df['n_trials']
df['stage'] = df['stage'].astype(str)

STAGE_ORDER = ['4.2', '4.3', '4.4', '4.5', '4.6', '4.7', '4.8']
stages_present = [s for s in STAGE_ORDER if s in df['stage'].unique()]
models = sorted(df['model'].unique())

# Mean and SEM of wbic_per_trial per model × stage
agg = (
    df.groupby(['model', 'stage'])['wbic_per_trial']
    .agg(mean='mean', sem=lambda x: x.std(ddof=1) / np.sqrt(len(x)))
    .reset_index()
)
agg['stage'] = pd.Categorical(agg['stage'], categories=STAGE_ORDER, ordered=True)
agg = agg.sort_values(['model', 'stage'])

# ── Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

for model in models:
    d = agg[agg['model'] == model]
    ax.errorbar(
        d['stage'].astype(str), d['mean'], yerr=d['sem'],
        marker='o', markersize=5, linewidth=1.8, capsize=3, label=model
    )

ax.set_xlabel('Training stage', fontsize=12)
ax.set_ylabel('Mean WBIC / trial', fontsize=12)
ax.set_title('Average WBIC per trial across training stages', fontsize=13)
ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()

os.makedirs(FIGURES_DIR, exist_ok=True)
_save_path = os.path.join(FIGURES_DIR, 'wbic_per_trial_by_stage.png')
plt.savefig(_save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved to {_save_path}')


In [ ]:
# ── Cell 9: Wmf and Wmb over sessions — per subject ──────────────────────
#
# X axis  : sessions in chronological order across stages 4.2 → 4.8
# Y axis  : raw Wmf (#E73F5F) and Wmb (#4C7FE3)
# Dividers: vertical dashed line at every stage boundary (aligned with ticks)
#
# One figure per subject, saved individually

import glob
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FixedFormatter, NullFormatter
from matplotlib.lines import Line2D

EPS = 1e-9

# ── Colours & style ────────────────────────────────────────────────────────
WMF_COLOR  = "#E73F5F"
WMB_COLOR  = "#4C7FE3"
STAGE_LINE = dict(color="k", linestyle="--", linewidth=0.9, alpha=0.6)
STAGE_ORDER = ['4.2', '4.3', '4.4', '4.5', '4.6', '4.7', '4.8']

# ── 0. Load & sort data ────────────────────────────────────────────────────
os.makedirs(FIGURES_DIR, exist_ok=True)

dfs = []
for f in sorted(glob.glob(os.path.join(WBIC_RESULTS_DIR, 'WBIC_hyb_p_*.csv'))):
    dfs.append(pd.read_csv(f))
df = pd.concat(dfs, ignore_index=True)
df['stage'] = df['stage'].astype(str)
df['date']  = pd.to_datetime(df['date'])

_stage_rank = {s: i for i, s in enumerate(STAGE_ORDER)}
df['_srank'] = df['stage'].map(_stage_rank)
df = df.sort_values(['subject', '_srank', 'date']).drop(columns='_srank')

SUBJECTS = sorted(df['subject'].unique())

_legend_handles = [
    Line2D([0], [0], color=WMF_COLOR, linewidth=2.4, label='Wmf'),
    Line2D([0], [0], color=WMB_COLOR, linewidth=2.4, label='Wmb'),
]

# ── Helper: build arrays for one subject ──────────────────────────────────
def build_subj_arrays(df_subj):
    rows = df_subj.reset_index(drop=True)
    x    = np.arange(len(rows))
    wmf  = rows['Wmf'].values.astype(float)
    wmb  = rows['Wmb'].values.astype(float)
    boundaries, prev = [], None
    for i, stage in enumerate(rows['stage']):
        if stage != prev:
            boundaries.append((i, stage))
            prev = stage
    return x, wmf, wmb, boundaries

# ── One figure per subject ─────────────────────────────────────────────────
for subj in SUBJECTS:
    df_s = df[df['subject'] == subj]
    x, wmf, wmb, bounds = build_subj_arrays(df_s)

    fig, ax = plt.subplots(figsize=(4, 3), dpi=300)

    # Stage-divider lines — all boundaries including the first (4.2)
    for xpos, _ in bounds:
        ax.axvline(xpos, **STAGE_LINE, zorder=0)

    # Parameter traces
    ax.plot(x, wmf, color=WMF_COLOR, linewidth=1.8, zorder=2)
    ax.plot(x, wmb, color=WMB_COLOR, linewidth=1.8, zorder=2)

    # Major ticks: stage boundaries (tick marks only, no labels)
    tick_x      = [xp for xp, _ in bounds]
    ax.xaxis.set_major_locator(FixedLocator(tick_x))
    ax.xaxis.set_major_formatter(NullFormatter())

    # Minor ticks: every session (marks only, no labels)
    ax.xaxis.set_minor_locator(FixedLocator(x))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.tick_params(axis='x', which='minor', length=3, width=0.7)

    ax.set_xlim(-0.5, len(x) - 0.5)
    ax.tick_params(axis='y', labelsize=11)
    ax.spines[['top', 'right']].set_visible(False)

    fig.tight_layout()
    out_path = os.path.join(FIGURES_DIR, f'hyb_p_Wmf_Wmb_{subj}.png')
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_path}')

In [ ]:
# ── Cell 10: Wmf and Wmb summary — per-stage mean ± SEM ───────────────────
#
# X axis: training stage (4.2 → 4.8)
# Y axis: raw Wmf (#E73F5F) and Wmb (#4C7FE3)
# Foreground: mean ± SEM across subjects, background: individual subjects

# Step 1: per-subject × per-stage mean of raw Wmf / Wmb
subj_stage_rows = []
for subj in SUBJECTS:
    df_s = df[df['subject'] == subj]
    x_s, wmf_s, wmb_s, _ = build_subj_arrays(df_s)
    stage_arr = df_s.reset_index(drop=True)['stage'].values
    for stage in STAGE_ORDER:
        mask = stage_arr == stage
        if not mask.any():
            continue
        subj_stage_rows.append({
            'subject': subj,
            'stage'  : stage,
            'Wmf'    : np.nanmean(wmf_s[mask]),
            'Wmb'    : np.nanmean(wmb_s[mask]),
        })

ss = pd.DataFrame(subj_stage_rows)

def _sem(v):
    return v.std(ddof=1) / np.sqrt(len(v))

agg = (
    ss.groupby('stage')[['Wmf', 'Wmb']]
      .agg(['mean', _sem])
      .reset_index()
)
agg.columns = ['stage', 'Wmf_mean', 'Wmf_sem', 'Wmb_mean', 'Wmb_sem']
agg['_ord'] = agg['stage'].map(_stage_rank)
agg = agg.sort_values('_ord').reset_index(drop=True)

x_sum      = np.arange(len(agg))
stg_labels = agg['stage'].values

fig_sum, ax_sum = plt.subplots(figsize=(4, 3), dpi=300)

# Background: individual subject trajectories
for subj in SUBJECTS:
    sub = ss[ss['subject'] == subj].copy()
    sub['_ord'] = sub['stage'].map(_stage_rank)
    sub = sub.sort_values('_ord')
    xi = sub['_ord'].values
    ax_sum.plot(xi, sub['Wmf'].values,
                color=WMF_COLOR, linewidth=0.8, alpha=0.35, zorder=1)
    ax_sum.plot(xi, sub['Wmb'].values,
                color=WMB_COLOR, linewidth=0.8, alpha=0.35, zorder=1)

# Mean traces + SEM error bars
ax_sum.errorbar(x_sum, agg['Wmf_mean'], yerr=agg['Wmf_sem'],
                color=WMF_COLOR, linewidth=2.4,
                marker='o', markersize=7, capsize=4, elinewidth=1.5,
                label='Wmf', zorder=3)
ax_sum.errorbar(x_sum, agg['Wmb_mean'], yerr=agg['Wmb_sem'],
                color=WMB_COLOR, linewidth=2.4,
                marker='o', markersize=7, capsize=4, elinewidth=1.5,
                label='Wmb', zorder=3)

ax_sum.xaxis.set_major_locator(FixedLocator(x_sum))
ax_sum.xaxis.set_major_formatter(FixedFormatter(stg_labels))
plt.setp(ax_sum.get_xticklabels(), rotation=45, ha='center', fontsize=11)
ax_sum.tick_params(axis='y', labelsize=11)
ax_sum.set_xlabel('Training stage', fontsize=11)
ax_sum.set_ylabel('Weight', fontsize=11)
ax_sum.set_title('hyb_p — Wmf and Wmb summary (mean ± SEM, n=6)',
                 fontsize=11)
ax_sum.spines[['top', 'right']].set_visible(False)
ax_sum.legend(handles=_legend_handles, frameon=False, fontsize=9)
fig_sum.tight_layout()
_out_sum = os.path.join(FIGURES_DIR, 'hyb_p_Wmf_Wmb_summary.png')
fig_sum.savefig(_out_sum, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {_out_sum}')


In [ ]:
# ── Cell 11: WBIC mean per stage (across sessions, across mice) → CSV ─────
#
# Builds a tidy table with one row per (model × stage):
#   - n_sessions, n_mice, n_trials_total
#   - WBIC_mean / WBIC_std            (raw WBIC across sessions)
#   - wbic_per_trial_mean / _std      (pooled across all sessions)
#   - wbic_per_trial_mice_mean / _std / _sem (per-subject mean → across mice)
#
# Also saves a wide-format stage-vs-model pivot with "mean ± std" strings.

import os, sys, glob
import numpy as np
import pandas as pd

# Self-contained path setup so this cell runs without first executing Cell 2.
_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    _organized = os.path.dirname(_here)
else:
    _organized = _here
if _organized not in sys.path:
    sys.path.insert(0, _organized)

from config import WBIC_RESULTS_DIR

STAGE_ORDER = ['4.2', '4.3', '4.4', '4.5', '4.6', '4.7', '4.8']

# Load all per-session CSVs (exclude subj/norm summaries)
_csv_files = sorted(glob.glob(os.path.join(WBIC_RESULTS_DIR, 'WBIC_*.csv')))
_csv_files = [f for f in _csv_files
              if not os.path.basename(f).startswith('WBIC_subj_')
              and not os.path.basename(f).startswith('WBIC_norm_')
              and not os.path.basename(f).startswith('WBIC_stage_')]

if not _csv_files:
    raise FileNotFoundError(
        f'No per-session WBIC_*.csv files found in {WBIC_RESULTS_DIR}. '
        'Run 04_wbic_fitting.ipynb first.'
    )

df_all_stages = pd.concat([pd.read_csv(f) for f in _csv_files], ignore_index=True)
df_all_stages['stage'] = df_all_stages['stage'].astype(str)
df_all_stages['wbic_per_trial'] = df_all_stages['WBIC'] / df_all_stages['n_trials']

# Per-subject mean of wbic_per_trial within each (model, stage)
_subj_mean = (
    df_all_stages
    .groupby(['model', 'stage', 'subject'])['wbic_per_trial']
    .mean()
    .reset_index()
)

def _sem_ddof1(v):
    v = v.dropna()
    return v.std(ddof=1) / np.sqrt(len(v)) if len(v) > 1 else np.nan

mice_agg = (
    _subj_mean
    .groupby(['model', 'stage'])['wbic_per_trial']
    .agg(
        wbic_per_trial_mice_mean='mean',
        wbic_per_trial_mice_std='std',
        wbic_per_trial_mice_sem=_sem_ddof1,
    )
    .reset_index()
)

sess_agg = (
    df_all_stages
    .groupby(['model', 'stage'])
    .agg(
        n_sessions=('WBIC', 'count'),
        n_mice=('subject', 'nunique'),
        n_trials_total=('n_trials', 'sum'),
        WBIC_mean=('WBIC', 'mean'),
        WBIC_std=('WBIC', 'std'),
        wbic_per_trial_mean=('wbic_per_trial', 'mean'),
        wbic_per_trial_std=('wbic_per_trial', 'std'),
    )
    .reset_index()
)

stage_summary = sess_agg.merge(mice_agg, on=['model', 'stage'])
stage_summary['stage'] = pd.Categorical(
    stage_summary['stage'], categories=STAGE_ORDER, ordered=True
)
stage_summary = stage_summary.sort_values(['model', 'stage']).reset_index(drop=True)

stage_summary['wbic_per_trial_mean_pm_std'] = (
    stage_summary['wbic_per_trial_mean'].round(4).astype(str) + ' ± ' +
    stage_summary['wbic_per_trial_std'].round(4).astype(str)
)
stage_summary['wbic_per_trial_mice_mean_pm_std'] = (
    stage_summary['wbic_per_trial_mice_mean'].round(4).astype(str) + ' ± ' +
    stage_summary['wbic_per_trial_mice_std'].round(4).astype(str)
)

_long_path = os.path.join(WBIC_RESULTS_DIR, 'WBIC_stage_summary.csv')
stage_summary.to_csv(_long_path, index=False)
print(f'Saved long-format summary: {_long_path}')

pivot_str = stage_summary.pivot(
    index='stage', columns='model', values='wbic_per_trial_mice_mean_pm_std'
).reindex(STAGE_ORDER)
_wide_path = os.path.join(WBIC_RESULTS_DIR, 'WBIC_stage_summary_wide.csv')
pivot_str.to_csv(_wide_path)
print(f'Saved wide-format pivot:    {_wide_path}')

print()
print('Stage × model WBIC/trial — across mice (mean ± std):')
print(pivot_str.to_string())
